# MIMIC-III: load the official open demo

This notebook accompanies Roger Mark's chapter [The Story of MIMIC](https://doi.org/10.1007/978-3-319-43742-2_5). It loads three small CSV tables from the official, unrestricted **MIMIC-III Clinical Database Demo v1.4** on PhysioNet.

The full MIMIC-III database requires PhysioNet credentialing, approved training, and a Data Use Agreement. The open demo contains 100 patients, excludes free-text notes, and should not be treated as representative of the full cohort.

In [ ]:
import json
import pandas as pd

BASE_URL = "https://physionet.org/files/mimiciii-demo/1.4"
TABLE_URLS = {
    "patients": f"{BASE_URL}/PATIENTS.csv",
    "admissions": f"{BASE_URL}/ADMISSIONS.csv",
    "icu_stays": f"{BASE_URL}/ICUSTAYS.csv",
}
TABLE_URLS

## Load three relational tables

These files total only tens of kilobytes. `subject_id` identifies a patient, `hadm_id` a hospital admission, and `icustay_id` an ICU stay.

In [ ]:
patients = pd.read_csv(TABLE_URLS["patients"])
admissions = pd.read_csv(TABLE_URLS["admissions"])
icu_stays = pd.read_csv(TABLE_URLS["icu_stays"])

print("PATIENTS:  ", patients.shape)
print("ADMISSIONS:", admissions.shape)
print("ICUSTAYS:   ", icu_stays.shape)

## Validate keys and join

A patient may have multiple admissions, and an admission may have multiple ICU stays. The assertions check the expected parent keys before a many-to-one join.

In [ ]:
assert patients["subject_id"].is_unique
assert admissions["hadm_id"].is_unique
assert icu_stays["icustay_id"].is_unique
assert set(admissions["subject_id"]).issubset(set(patients["subject_id"]))
assert set(icu_stays["hadm_id"]).issubset(set(admissions["hadm_id"]))

joined = (
    icu_stays
    .merge(admissions, on=["subject_id", "hadm_id"], how="left", validate="many_to_one")
    .merge(patients, on="subject_id", how="left", validate="many_to_one")
)

preview_columns = [
    "subject_id", "hadm_id", "icustay_id", "gender",
    "admission_type", "first_careunit", "intime", "outtime", "los",
]
joined[preview_columns].head(2)

## Convert two rows to a compact nested JSON format

MIMIC is distributed as related tables rather than one canonical JSON record. This helper creates a convenient per-ICU-stay view while preserving the important identifiers.

In [ ]:
def clean(value):
    return None if pd.isna(value) else value.item() if hasattr(value, "item") else value

def to_nested_record(row):
    return {
        "patient": {
            "subject_id": clean(row["subject_id"]),
            "gender": clean(row["gender"]),
            "expire_flag": clean(row["expire_flag"]),
        },
        "admission": {
            "hadm_id": clean(row["hadm_id"]),
            "admittime": clean(row["admittime"]),
            "dischtime": clean(row["dischtime"]),
            "admission_type": clean(row["admission_type"]),
            "admission_location": clean(row["admission_location"]),
            "diagnosis": clean(row["diagnosis"]),
            "hospital_expire_flag": clean(row["hospital_expire_flag"]),
        },
        "icu_stay": {
            "icustay_id": clean(row["icustay_id"]),
            "first_careunit": clean(row["first_careunit"]),
            "last_careunit": clean(row["last_careunit"]),
            "intime": clean(row["intime"]),
            "outtime": clean(row["outtime"]),
            "los_days": clean(row["los"]),
        },
    }

records = [to_nested_record(row) for _, row in joined.head(2).iterrows()]
print(json.dumps(records, indent=2))

## Next steps

The same identifiers can join other open demo tables such as diagnoses, laboratory events, and chart events. For work using the full MIMIC database, obtain access from PhysioNet and review its current data-use and online-service requirements before moving any data into a notebook environment.